[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/07-interview-prep/02_python-advanced.ipynb)

*Interview Prep · Notebook 2 of 5*

# Advanced Python

The Python that separates seniors: closures and late binding, decorators, generators and iterators, context managers, copying, the MRO, dataclasses, standard-library tools, and concurrency concepts.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A name is a label on an object, and functions are objects too. Decorators, closures and generators all follow from those two facts.

**What interviewers really test here:** Common traps, writing a decorator from nothing, generators vs lists, and explaining the GIL.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Names, objects and the traps they cause *(Basics)*
- [ ] 2. Decorators, from nothing *(Core)*
- [ ] 3. Iterators and generators *(Core)*
- [ ] 4. Context managers and classes *(Core)*
- [ ] 5. Know your standard library *(Core)*
- [ ] 6. Concurrency: GIL, threads, processes and async *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Memory in CPython

- **Reference counting:** each object tracks how many labels point at it, and it's freed at zero.
- **Garbage collector:** finds reference cycles that counting alone misses.
- **`__slots__`:** removes each object's `__dict__`, saving memory when you have many small objects.
- **Everything is an object,** which is why a Python list of numbers uses far more memory than a NumPy array.

### Mutable vs immutable

| Immutable | Mutable |
| --- | --- |
| int, float, str, tuple, frozenset, bytes | list, dict, set, most objects |



Only hashable values, usually immutable ones, can be dictionary keys or set members. A tuple holding a list isn't hashable.

---
# Part 2: Lessons, step by step

## Lesson 1: Names, objects and the traps they cause

*Level: Basics*

A variable is a label on an object. Assignment never copies; it adds another label.



```python
a = [1, 2]
b = a          # the same list, with two labels
b.append(3)
print(a)       # [1, 2, 3]
```



**`==` vs `is`:** `==` asks "same value?" and `is` asks "same object?". Use `is` only for None. Small integers and some strings are cached, so `is` sometimes appears to work on them, and that luck is unreliable.



**Trap 1: mutable default arguments.** A default is created once, when the function is defined:



```python
def add(x, items=[]):    # the same list on every call!
    items.append(x)
    return items
add(1)   # [1]
add(2)   # [1, 2]  surprise
```



The fix is `items=None`, then `items = [] if items is None else items`.



**Trap 2: late binding.** A closure remembers the *variable*, not its value at that moment:



```python
fns = [lambda v: v * i for i in range(3)]
[f(10) for f in fns]     # [20, 20, 20], because i is 2 by the time they run
```



The fix is `lambda v, i=i: v * i`, which gives [0, 10, 20].



**Trap 3: shallow copies.** After `b = copy.copy(a)` on a nested list, the outer list is new but the inner lists are shared:



| Copy | Change a[0][0] = 99, then b[0] is |
| --- | --- |
| `b = a` | [99, 2] |
| `copy.copy(a)` | [99, 2] |
| `copy.deepcopy(a)` | [1, 2] |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def late_binding_demo():
    wrong = [lambda v: v * i for i in range(3)]
    right = [lambda v, i=i: v * i for i in range(3)]
    print("late binding:", [f(10) for f in wrong])
    print("fixed:       ", [f(10) for f in right])


late_binding_demo()

late binding: [20, 20, 20]
fixed:        [0, 10, 20]


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q9](#q9), [Q5](#q5), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: Decorators, from nothing

*Level: Core*

Functions are values, so you can pass one in and return another. A **decorator** is a function that takes a function and returns an improved version.



```python
def shout(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

@shout
def greet(name):
    return "hi " + name

greet("sam")     # 'HI SAM'
```



The `@shout` line is only shorthand for `greet = shout(greet)`. There's no magic beyond that.



**Always use `functools.wraps`.** Otherwise the decorated function reports its name as "wrapper", which confuses debugging, logging and documentation. `wraps` copies the name and docstring across.



**A decorator with arguments has three levels,** because `@retry(3)` must first run and *return* a decorator:



```python
def retry(times):            # 1. takes the settings
    def deco(fn):            # 2. takes the function
        @functools.wraps(fn)
        def wrapper(*a, **k):    # 3. runs at each call
            for attempt in range(times):
                try:
                    return fn(*a, **k)
                except ValueError:
                    if attempt == times - 1:
                        raise
        return wrapper
    return deco
```



**Memoisation shows the payoff.** Plain recursive fib(30) makes 2,692,537 calls. With `@functools.lru_cache`, the body runs just **31** times, once for each of 0 to 30.



Common real uses are logging, timing, retries, caching, access checks, and registering routes in web frameworks.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q10](#q10), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 3: Iterators and generators

*Level: Core*

A `for` loop actually does this: call `iter()` on the object, then call `next()` until `StopIteration`.



**Iterable vs iterator:** a list is iterable, so you can loop over it many times. An iterator is a one-way stream with a position: once used up, it stays empty.



A **generator** is the easiest way to make an iterator. `yield` pauses the function and keeps all its local variables, and the next `next()` call resumes it.



```python
def countdown(n):
    while n > 0:
        yield n        # pause here, handing n out
        n -= 1

c = countdown(3)
list(c)    # [3, 2, 1]
list(c)    # []  used up
```



**Why it matters: memory.** A list of a million numbers takes megabytes. A generator holds only its current state, whatever the length, so it can process a file larger than your RAM, or an endless stream.



**Pipelines** chain lazily, and nothing happens until the last step asks for a value:



```python
lines = (l.strip() for l in open(path))
kept = (l for l in lines if l and not l.startswith("#"))
for line in kept: ...
```



**`yield from`** hands over to another generator, which makes recursive ones short.



The demo measures the difference in memory.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def generator_memory():
    import sys
    as_list = [i * i for i in range(100000)]
    as_generator = (i * i for i in range(100000))
    print("list bytes:     ", sys.getsizeof(as_list))
    print("generator bytes:", sys.getsizeof(as_generator))
    print("same total:     ", sum(as_list) == sum(as_generator))


generator_memory()

list bytes:      800984
generator bytes: 208
same total:      True


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q14](#q14), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 4: Context managers and classes

*Level: Core*

**A context manager** guarantees clean-up. `with` calls `__enter__` at the start and `__exit__` at the end, even if an error occurs. If `__exit__` returns True, the error is suppressed.



```python
with open("a.txt") as f:     # closed for you, even on an error
    data = f.read()
```



Use one wherever something must be released: files, locks, database connections. `contextlib.contextmanager` makes one from a generator with a single yield.



**Multiple inheritance and the MRO.** For the diamond `class D(B, C)` where B and C inherit from A, the method resolution order is **D, B, C, A, object**. `super()` means "the next class in the MRO", not "my parent". That's why each class's method runs exactly once, giving "DBCA".



**`__eq__` and `__hash__` go together.** Sets and dictionaries find an item by hash, then confirm with equality. If equal objects had different hashes, the set would keep both. Defining `__eq__` alone makes a class unhashable, on purpose. `@dataclass(frozen=True)` generates both correctly.



| Method kind | First parameter | Use for |
| --- | --- | --- |
| Instance | self | work with one object's data |
| `@classmethod` | cls | alternative constructors |
| `@staticmethod` | nothing | a helper that lives with the class |
| `@property` | self | a computed or validated attribute |

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 5: Know your standard library

*Level: Core*

Interviewers notice when you reach for the right tool, because it means less code and fewer bugs.



| Tool | Does | Instead of |
| --- | --- | --- |
| `Counter(words).most_common(3)` | counts, top k | a dict and a sort |
| `defaultdict(list)` | grouping | "if key not in d" |
| `deque` | O(1) at both ends | `list.pop(0)`, which is O(n) |
| `heapq.nlargest(k, xs)` | top k in O(n log k) | a full sort |
| `bisect` | binary search on sorted lists | writing it by hand |
| `OrderedDict` | LRU with `move_to_end` | a linked list by hand |
| `itertools.groupby` | runs of neighbours | tracking "previous" |
| `functools.lru_cache` | memoisation | a dict by hand |



**Trap:** `groupby` only groups *neighbours*. "abca" gives a, b, c, a separately, so sort first if you want one group per key.



**Costs worth memorising:**



| Operation | Cost |
| --- | --- |
| `list.append`, `dict[k]`, `x in set` | O(1) (average or amortised) |
| `x in list`, `list.insert(0, x)`, `list.pop(0)` | O(n) |
| `sorted` (Timsort, stable) | O(n log n) |
| Repeated `s += "x"` in a loop | O(n²) worst case, so use `"".join` |

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q6](#q6), [Q7](#q7), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 6: Concurrency: GIL, threads, processes and async

*Level: Advanced*

"What is the GIL?" is the most common advanced Python question.



**The Global Interpreter Lock** lets only one thread run Python bytecode at a time in standard CPython. So threads don't speed up pure-Python number crunching. They do help when waiting, because the lock is released during I/O, and many libraries like NumPy release it during heavy work.



| Tool | Best for | Why |
| --- | --- | --- |
| `threading` | I/O: network, files | threads wait without blocking each other |
| `multiprocessing` | CPU-heavy work | separate processes, each with its own GIL |
| `asyncio` | many thousands of waiting connections | one thread, cheap task switching at each `await` |



```python
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

with ThreadPoolExecutor(max_workers=8) as pool:
    pages = list(pool.map(fetch, urls))         # I/O: threads

with ProcessPoolExecutor() as pool:
    results = list(pool.map(crunch, chunks))    # CPU: processes
```



```python
import asyncio

async def fetch(name, delay):
    await asyncio.sleep(delay)    # gives way to other tasks while waiting
    return name

async def main():
    return await asyncio.gather(fetch("a", 0.2), fetch("b", 0.1))

print(asyncio.run(main()))   # ['a', 'b'], in about 0.2 seconds, not 0.3
```



These snippets are shown as text, as they don't suit this site's browser IDE. Run them on your machine or in Colab (where a notebook's event loop means you'd write `await main()` instead of `asyncio.run`).



**What async really is.** Tasks are paused functions, and a loop runs each until it next waits, which is the "Coroutines by Hand" question with generators. Note that one blocking call, like `time.sleep`, freezes every task.



**Race condition:** `counter += 1` is several steps (read, add, write), so threads can overwrite each other. Protect shared data with a `Lock`, or better, don't share: pass messages with a `queue.Queue`.



**Good to mention:** Python 3.13 added an optional free-threaded build without the GIL. It's still not the default, so assume the GIL unless told otherwise.

**✏️ Your turn:** you now have what you need for [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [4]:
import functools

from collections import Counter, OrderedDict, deque


<a id="q1"></a>
### Q1. Mutable Default Argument

🟢 Warm-up · *The classic trap*

Write append\_to(value, target=None) so each call without a target starts with a new list. Test: run calls [(1,), (2,)] and return results.

**Examples**

- `calls = [(1,), (2,)]` → `[[1], [2]]`

In [ ]:
def default_trap(calls):
    """
    Write append_to(value, target=None) so each call without a target starts
    with a new list. Test: run calls [(1,), (2,)] and return results.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(default_trap, [
    (([(1,), (2,)],), [[1], [2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Without the fix, a default list is shared between calls. Check for None and build a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same idea written compactly. None is the standard sentinel.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Without the fix, a default list is shared between calls. Check for None and build a list.

```python
def default_trap(calls):
    def append_to(value, target=None):
        if target is None:
            target = []
        target.append(value)
        return target
    results = []
    for c in calls:
        results.append(append_to(*c))
    return results
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same idea written compactly. None is the standard sentinel.

```python
def default_trap(calls):
    def append_to(value, target=None):
        target = [] if target is None else target
        target.append(value)
        return target
    return [append_to(*c) for c in calls]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Without the fix, a default list is shared between calls. Check for None and build a list. That costs O(n) time and O(n) space."

"I can do better: Same idea written compactly. None is the standard sentinel. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `calls = [(1,), (2,)]`

1. append\_to() returns [1]
2. append\_to() returns [2]
3. default\_trap() returns [[1], [2]]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/mutable-default-argument)

<a id="q2"></a>
### Q2. A Closure With State

🟢 Warm-up · *nonlocal*

Make a counter function starting at start. Call it n times and return the results.

**Examples**

- `start = 5, n = 3` → `[6, 7, 8]`

In [ ]:
def counter_calls(start, n):
    """
    Make a counter function starting at start. Call it n times and return the
    results.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(counter_calls, [
    ((5, 3), [6, 7, 8]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class holding the count.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A closure with `nonlocal`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A class holding the count.

```python
def counter_calls(start, n):
    class Counter_:
        def __init__(self):
            self.value = start

        def __call__(self):
            self.value += 1
            return self.value
    c = Counter_()
    return [c() for _ in range(n)]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A closure with `nonlocal`.

```python
def counter_calls(start, n):
    def make():
        value = start

        def next_value():
            nonlocal value
            value += 1
            return value
        return next_value
    c = make()
    return [c() for _ in range(n)]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class holding the count. That costs O(n) time and O(1) space."

"I can do better: A closure with `nonlocal`. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `start = 5, n = 3`

1. make() returns &lt;function counter\_calls.&lt;locals&gt;.make...
2. next\_value() returns 6
3. next\_value() returns 7
4. next\_value() returns 8
5. counter\_calls() returns [6, 7, 8]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/a-closure-with-state)

<a id="q3"></a>
### Q3. Count-Calls Decorator

🟢 Warm-up · *Your first decorator*

Write a decorator that counts calls in wrapper.calls. Apply it to a square function, call it on each value, and return (results, count).

**Examples**

- `values = [2, 3, 4]` → `([4, 9, 16], 3)`

In [ ]:
def count_calls(values):
    """
    Write a decorator that counts calls in wrapper.calls. Apply it to a square
    function, call it on each value, and return (results, count).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_calls, [
    (([2, 3, 4],), ([4, 9, 16], 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Wrap it by hand, without @.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`@` syntax with `functools.wraps`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Wrap it by hand, without @.

```python
def count_calls(values):
    def counted(fn):
        def wrapper(*args):
            wrapper.calls += 1
            return fn(*args)
        wrapper.calls = 0
        return wrapper

    def square(x):
        return x * x
    square = counted(square)
    return ([square(v) for v in values], square.calls)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `@` syntax with `functools.wraps`.

```python
def count_calls(values):
    def counted(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            wrapper.calls += 1
            return fn(*args, **kwargs)
        wrapper.calls = 0
        return wrapper

    @counted
    def square(x):
        return x * x
    return ([square(v) for v in values], square.calls)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Wrap it by hand, without @. That costs O(n) time and O(n) space."

"I can do better: `@` syntax with `functools.wraps`. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `values = [2, 3, 4]`

1. counted() returns &lt;function count\_calls.&lt;locals&gt;.square...
2. square() returns 4
3. wrapper() returns 4
4. square() returns 9
5. wrapper() returns 9
6. square() returns 16
7. wrapper() returns 16
8. count\_calls() returns ([4, 9, 16], 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/count-calls-decorator)

<a id="q4"></a>
### Q4. Why functools.wraps Matters

🟢 Warm-up · *Keeping the name*

Return (name without wraps, name with wraps) of a decorated function called greet.

**Examples**

- `` → `('wrapper', 'greet')`

In [ ]:
def wraps_demo():
    """
    Return (name without wraps, name with wraps) of a decorated function called
    greet.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wraps_demo, [
    ((), ('wrapper', 'greet')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Set `wrapper.__name__` by hand in the second version.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`functools.wraps` copies the name, docstring and more.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Set `wrapper.__name__` by hand in the second version.

```python
def wraps_demo():
    def plain_deco(fn):
        def wrapper():
            return fn()
        return wrapper

    def named_deco(fn):
        def wrapper():
            return fn()
        wrapper.__name__ = fn.__name__
        return wrapper

    def greet():
        return "hi"
    return (plain_deco(greet).__name__, named_deco(greet).__name__)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `functools.wraps` copies the name, docstring and more.

```python
def wraps_demo():
    def plain_deco(fn):
        def wrapper():
            return fn()
        return wrapper

    def named_deco(fn):
        @functools.wraps(fn)
        def wrapper():
            return fn()
        return wrapper

    def greet():
        return "hi"
    return (plain_deco(greet).__name__, named_deco(greet).__name__)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Set `wrapper.__name__` by hand in the second version. That costs O(1) time and O(1) space."

"I can do better: `functools.wraps` copies the name, docstring and more. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: ``

1. plain\_deco() returns &lt;function wraps\_demo.&lt;locals&gt;.plain\_d...
2. named\_deco() returns &lt;function wraps\_demo.&lt;locals&gt;.greet a...
3. wraps\_demo() returns ('wrapper', 'greet')

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/why-functools-wraps-matters)

<a id="q5"></a>
### Q5. Shallow vs Deep Copy

🟢 Warm-up · *Aliasing bugs*

Copy a nested list three ways (assignment, shallow, deep), then change original[0][0] to 99. Return each copy's first inner list.

**Examples**

- `original = [[1, 2], [3]]` → `([99, 2], [99, 2], [1, 2])`

In [ ]:
import copy

def copy_kinds(original):
    """
    Copy a nested list three ways (assignment, shallow, deep), then change
    original[0][0] to 99. Return each copy's first inner list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(copy_kinds, [
    (([[1, 2], [3]],), ([99, 2], [99, 2], [1, 2])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the shallow and deep copies by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`copy.copy` and `copy.deepcopy`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build the shallow and deep copies by hand.

```python
def copy_kinds(original):
    alias = original
    shallow = [row for row in original]
    deep = [[v for v in row] for row in original]
    original[0][0] = 99
    return (alias[0], shallow[0], deep[0])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `copy.copy` and `copy.deepcopy`.

```python
def copy_kinds(original):
    alias, shallow, deep = original, copy.copy(original), copy.deepcopy(original)
    original[0][0] = 99
    return (alias[0], shallow[0], deep[0])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the shallow and deep copies by hand. That costs O(n) time and O(n) space."

"I can do better: `copy.copy` and `copy.deepcopy`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/shallow-vs-deep-copy)

<a id="q6"></a>
### Q6. Group Consecutive Runs

🟢 Warm-up · *itertools.groupby*

Compress 'aaabbc' into [('a', 3), ('b', 2), ('c', 1)]. Note groupby only groups neighbours.

**Examples**

- `text = 'aaabbc'` → `[('a', 3), ('b', 2), ('c', 1)]`
- `text = 'abca'` → `[('a', 1), ('b', 1), ('c', 1), ('a', 1)]`

In [ ]:
import itertools

def runs(text):
    """
    Compress 'aaabbc' into [('a', 3), ('b', 2), ('c', 1)]. Note groupby only
    groups neighbours.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(runs, [
    (('aaabbc',), [('a', 3), ('b', 2), ('c', 1)]),
    (('abca',), [('a', 1), ('b', 1), ('c', 1), ('a', 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the current character in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.groupby`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Track the current character in a loop.

```python
def runs(text):
    out = []
    for ch in text:
        if out and out[-1][0] == ch:
            out[-1] = (ch, out[-1][1] + 1)
        else:
            out.append((ch, 1))
    return out
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.groupby`.

```python
def runs(text):
    return [(k, len(list(g))) for k, g in itertools.groupby(text)]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the current character in a loop. That costs O(n) time and O(k) space."

"I can do better: `itertools.groupby`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/group-consecutive-runs)

<a id="q7"></a>
### Q7. Top K With the Standard Library

🟢 Warm-up · *Counter, heapq*

Return the k most common words, as (word, count), ties alphabetical.

**Examples**

- `words = ['b', 'a', 'b', 'c', 'a', 'b'], k = 2` → `[('b', 3), ('a', 2)]`

In [ ]:
import heapq

def top_words(words, k):
    """
    Return the k most common words, as (word, count), ties alphabetical.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_words, [
    ((['b', 'a', 'b', 'c', 'a', 'b'], 2), [('b', 3), ('a', 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary, then a full sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`heapq.nsmallest` over a Counter is O(n log k).

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary, then a full sort.

```python
def top_words(words, k):
    counts = {}
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `heapq.nsmallest` over a Counter is O(n log k).

```python
def top_words(words, k):
    return heapq.nsmallest(k, Counter(words).items(), key=lambda kv: (-kv[1], kv[0]))
```

**Cost:** O(n log k) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary, then a full sort. That costs O(n log n) time and O(n) space."

"I can do better: `heapq.nsmallest` over a Counter is O(n log k). That's O(n log k) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/top-k-with-the-standard-library)

<a id="q8"></a>
### Q8. Keep a List Sorted

🟢 Warm-up · *bisect*

Insert each value into a sorted list, keeping it sorted, and return the final list and the position where target would go.

**Examples**

- `values = [5, 1, 3], target = 4` → `([1, 3, 5], 2)`

In [ ]:
import bisect

def sorted_insert(values, target):
    """
    Insert each value into a sorted list, keeping it sorted, and return the
    final list and the position where target would go.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sorted_insert, [
    (([5, 1, 3], 4), ([1, 3, 5], 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for the position each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`bisect` finds the position in O(log n), though inserting still shifts items.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Scan for the position each time.

```python
def sorted_insert(values, target):
    out = []
    for v in values:
        i = 0
        while i < len(out) and out[i] < v:
            i += 1
        out.insert(i, v)
    pos = 0
    while pos < len(out) and out[pos] < target:
        pos += 1
    return (out, pos)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `bisect` finds the position in O(log n), though inserting still shifts items.

```python
def sorted_insert(values, target):
    out = []
    for v in values:
        bisect.insort(out, v)
    return (out, bisect.bisect_left(out, target))
```

**Cost:** O(n²) worst, fast finds time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for the position each time. That costs O(n²) time and O(n) space."

"I can do better: `bisect` finds the position in O(log n), though inserting still shifts items. That's O(n²) worst, fast finds time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/keep-a-list-sorted)

<a id="q9"></a>
### Q9. Late-Binding Closures

🟡 Interview standard · *The loop variable trap*

Return a list of n functions where the i-th multiplies its argument by i, then apply each to x. A plain lambda in a loop would give n−1 every time.

**Examples**

- `n = 3, x = 10` → `[0, 10, 20]`

In [ ]:
def multipliers(n, x):
    """
    Return a list of n functions where the i-th multiplies its argument by i,
    then apply each to x. A plain lambda in a loop would give n−1 every time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(multipliers, [
    ((3, 10), [0, 10, 20]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A factory function captures each value in its own scope.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A default argument, `lambda v, i=i: v * i`, binds the value at creation.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A factory function captures each value in its own scope.

```python
def multipliers(n, x):
    def make(i):
        def f(v):
            return v * i
        return f
    fns = []
    for i in range(n):
        fns.append(make(i))
    return [f(x) for f in fns]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A default argument, `lambda v, i=i: v * i`, binds the value at creation.

```python
def multipliers(n, x):
    fns = [lambda v, i=i: v * i for i in range(n)]
    return [f(x) for f in fns]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A factory function captures each value in its own scope. That costs O(n) time and O(n) space."

"I can do better: A default argument, `lambda v, i=i: v * i`, binds the value at creation. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/late-binding-closures)

<a id="q10"></a>
### Q10. Memoize Decorator

🟡 Interview standard · *Build lru\_cache yourself*

Compute fib(n) with memoisation, and return (fib(n), number of times the real body ran).

**Examples**

- `n = 30` → `(832040, 31)`
- `n = 10` → `(55, 11)`

In [ ]:
def memo_fib(n):
    """
    Compute fib(n) with memoisation, and return (fib(n), number of times the
    real body ran).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(memo_fib, [
    ((30,), (832040, 31)),
    ((10,), (55, 11)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary-based decorator written by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`functools.lru_cache`, and `cache_info()` reports misses.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary-based decorator written by hand.

```python
def memo_fib(n):
    def memoize(fn):
        cache = {}

        def wrapper(x):
            if x not in cache:
                cache[x] = fn(x)
            return cache[x]
        wrapper.cache = cache
        return wrapper

    @memoize
    def fib(k):
        return k if k < 2 else fib(k - 1) + fib(k - 2)
    return (fib(n), len(fib.cache))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `functools.lru_cache`, and `cache_info()` reports misses.

```python
def memo_fib(n):
    @functools.lru_cache(maxsize=None)
    def fib(k):
        return k if k < 2 else fib(k - 1) + fib(k - 2)
    return (fib(n), fib.cache_info().misses)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary-based decorator written by hand. That costs O(n) time and O(n) space."

"I can do better: `functools.lru_cache`, and `cache_info()` reports misses. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 30`

1. fib() returns 1
2. fib() returns 0
3. fib() returns 1
4. fib() returns 2
5. fib() returns 3
6. fib() returns 5
7. fib() returns 8
8. fib() returns 13
9. fib() returns 21
10. fib() returns 34
11. fib() returns 55
12. fib() returns 89
13. fib() returns 144
14. fib() returns 233
15. fib() returns 377

*…and so on.*

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/memoize-decorator)

<a id="q11"></a>
### Q11. Decorator With Arguments

🟡 Interview standard · *Three levels of functions*

Write retry(times): it calls a function until it succeeds, re-raising after the last try. Apply it to a function that fails until its fails\_before-th call. Return (result or 'gave up', calls made).

**Examples**

- `times = 3, fails_before = 2` → `('ok', 3)`
- `times = 2, fails_before = 5` → `('gave up', 2)`

In [ ]:
def retry_demo(times, fails_before):
    """
    Write retry(times): it calls a function until it succeeds, re-raising after
    the last try. Apply it to a function that fails until its fails_before-th
    call. Return (result or 'gave up', calls made).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retry_demo, [
    ((3, 2), ('ok', 3)),
    ((2, 5), ('gave up', 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop tracking the last error.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `for` loop that re-raises on the last attempt, with `functools.wraps` and any arguments.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop tracking the last error.

```python
def retry_demo(times, fails_before):
    def retry(limit):
        def deco(fn):
            def wrapper():
                last = None
                count = 0
                while count < limit:
                    try:
                        return fn()
                    except ValueError as e:
                        last = e
                        count += 1
                raise last
            return wrapper
        return deco
    calls = [0]

    @retry(times)
    def flaky():
        calls[0] += 1
        if calls[0] <= fails_before:
            raise ValueError("not yet")
        return "ok"
    try:
        return (flaky(), calls[0])
    except ValueError:
        return ("gave up", calls[0])
```

**Cost:** O(t) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A `for` loop that re-raises on the last attempt, with `functools.wraps` and any arguments.

```python
def retry_demo(times, fails_before):
    def retry(limit):
        def deco(fn):
            @functools.wraps(fn)
            def wrapper(*args, **kwargs):
                for attempt in range(limit):
                    try:
                        return fn(*args, **kwargs)
                    except ValueError:
                        if attempt == limit - 1:
                            raise
            return wrapper
        return deco
    calls = [0]

    @retry(times)
    def flaky():
        calls[0] += 1
        if calls[0] <= fails_before:
            raise ValueError("not yet")
        return "ok"
    try:
        return (flaky(), calls[0])
    except ValueError:
        return ("gave up", calls[0])
```

**Cost:** O(t) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop tracking the last error. That costs O(t) time and O(1) space."

"I can do better: A `for` loop that re-raises on the last attempt, with `functools.wraps` and any arguments. That's O(t) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `times = 3, fails_before = 2`

1. retry() returns &lt;function retry\_demo.&lt;locals&gt;.retry.&lt;...
2. deco() returns &lt;function retry\_demo.&lt;locals&gt;.flaky a...
3. ValueError raised: not yet
4. flaky() returns None
5. ValueError raised: not yet
6. ValueError raised: not yet
7. flaky() returns None
8. ValueError raised: not yet
9. flaky() returns 'ok'
10. wrapper() returns 'ok'
11. retry\_demo() returns ('ok', 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/decorator-with-arguments)

<a id="q12"></a>
### Q12. Generator Pipeline

🟡 Interview standard · *Lazy steps*

Using generators, take lines, strip them, drop empty ones and those starting with #, and return the upper-cased remainder as a list.

**Examples**

- `lines = [' a ', '', '# note', 'b']` → `['A', 'B']`

In [ ]:
def clean_lines(lines):
    """
    Using generators, take lines, strip them, drop empty ones and those starting
    with #, and return the upper-cased remainder as a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clean_lines, [
    (([' a ', '', '# note', 'b'],), ['A', 'B']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One loop doing all the steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Three chained generator expressions. Nothing runs until consumed, so it can handle huge files.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** One loop doing all the steps.

```python
def clean_lines(lines):
    out = []
    for line in lines:
        s = line.strip()
        if s and not s.startswith("#"):
            out.append(s.upper())
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Three chained generator expressions. Nothing runs until consumed, so it can handle huge files.

```python
def clean_lines(lines):
    stripped = (l.strip() for l in lines)
    kept = (l for l in stripped if l and not l.startswith("#"))
    return list(l.upper() for l in kept)
```

**Cost:** O(n) time, O(1) per item space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One loop doing all the steps. That costs O(n) time and O(n) space."

"I can do better: Three chained generator expressions. Nothing runs until consumed, so it can handle huge files. That's O(n) time and O(1) per item space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/generator-pipeline)

<a id="q13"></a>
### Q13. Sliding Window Generator

🟡 Interview standard · *Streams of any length*

Yield tuples of the last k items of any iterable. It must work with an iterator that can't be indexed.

**Examples**

- `items = [1, 2, 3, 4], k = 2` → `[(1, 2), (2, 3), (3, 4)]`

In [ ]:
def sliding(items, k):
    """
    Yield tuples of the last k items of any iterable. It must work with an
    iterator that can't be indexed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sliding, [
    (([1, 2, 3, 4], 2), [(1, 2), (2, 3), (3, 4)]),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list, popping the front (O(k) each).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`deque(maxlen=k)` drops the oldest automatically.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Keep a list, popping the front (O(k) each).

```python
def sliding(items, k):
    window = []
    for x in items:
        window.append(x)
        if len(window) > k:
            window.pop(0)
        if len(window) == k:
            yield tuple(window)
```

**Cost:** O(n·k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `deque(maxlen=k)` drops the oldest automatically.

```python
def sliding(items, k):
    window = deque(maxlen=k)
    for x in items:
        window.append(x)
        if len(window) == k:
            yield tuple(window)
```

**Cost:** O(n·k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list, popping the front (O(k) each). That costs O(n·k) time and O(k) space."

"I can do better: `deque(maxlen=k)` drops the oldest automatically. That's O(n·k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `items = [1, 2, 3, 4], k = 2`

1. sliding() returns (1, 2)
2. sliding() returns (2, 3)
3. sliding() returns (3, 4)
4. sliding() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/sliding-window-generator)

<a id="q14"></a>
### Q14. Flatten With yield from

🟡 Interview standard · *Recursive generators*

Flatten nested lists of any depth, lazily.

**Examples**

- `items = [1, [2, [3, 4]], 5]` → `[1, 2, 3, 4, 5]`

In [ ]:
def flatten(items):
    """
    Flatten nested lists of any depth, lazily.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [
    (([1, [2, [3, 4]], 5],), [1, 2, 3, 4, 5]),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An inner loop re-yields each value.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`yield from` delegates to the inner generator.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** An inner loop re-yields each value.

```python
def flatten(items):
    for x in items:
        if isinstance(x, list):
            for y in flatten(x):
                yield y
        else:
            yield x
```

**Cost:** O(n·d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `yield from` delegates to the inner generator.

```python
def flatten(items):
    for x in items:
        if isinstance(x, list):
            yield from flatten(x)
        else:
            yield x
```

**Cost:** O(n·d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An inner loop re-yields each value. That costs O(n·d) time and O(d) space."

"I can do better: `yield from` delegates to the inner generator. That's O(n·d) time and O(d) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `items = [1, [2, [3, 4]], 5]`

1. flatten() returns 1
2. flatten() returns 2
3. flatten() returns 2
4. flatten() returns 3
5. flatten() returns 3
6. flatten() returns 3
7. flatten() returns 4
8. flatten() returns 4
9. flatten() returns 4
10. flatten() returns None
11. StopIteration raised: 
12. flatten() returns None
13. StopIteration raised: 
14. flatten() returns 5
15. flatten() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/flatten-with-yield-from)

<a id="q15"></a>
### Q15. Build an Iterator

🟡 Interview standard · *\_\_iter\_\_ and \_\_next\_\_*

Make Countdown(n), which can be looped over from n to 1. Return list(Countdown(n)), and also whether a second pass over the same object gives nothing.

**Examples**

- `n = 3` → `([3, 2, 1], True)`

In [ ]:
def countdown_demo(n):
    """
    Make Countdown(n), which can be looped over from n to 1. Return
    list(Countdown(n)), and also whether a second pass over the same object
    gives nothing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(countdown_demo, [
    ((3,), ([3, 2, 1], True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class with `__iter__` and `__next__`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator gives the same behaviour in two lines.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A class with `__iter__` and `__next__`.

```python
def countdown_demo(n):
    class Countdown:
        def __init__(self, start):
            self.current = start

        def __iter__(self):
            return self

        def __next__(self):
            if self.current <= 0:
                raise StopIteration
            self.current -= 1
            return self.current + 1
    c = Countdown(n)
    return (list(c), list(c) == [])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A generator gives the same behaviour in two lines.

```python
def countdown_demo(n):
    def countdown(start):
        while start > 0:
            yield start
            start -= 1
    c = countdown(n)
    return (list(c), list(c) == [])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class with `__iter__` and `__next__`. That costs O(n) time and O(1) space."

"I can do better: A generator gives the same behaviour in two lines. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 3`

1. countdown() returns 3
2. countdown() returns 2
3. countdown() returns 1
4. countdown() returns None
5. countdown\_demo() returns ([3, 2, 1], True)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/build-an-iterator)

<a id="q16"></a>
### Q16. A Context Manager

🟡 Interview standard · *\_\_enter\_\_ and \_\_exit\_\_*

Write a context manager that logs 'enter' and 'exit' even on error, and swallows only ValueError. For each flag in raises, run a with-block that raises ValueError if the flag is true. Return the log.

**Examples**

- `raises = [False, True]` → `['enter', 'exit', 'enter', 'exit']`

In [ ]:
def log_context(raises):
    """
    Write a context manager that logs 'enter' and 'exit' even on error, and
    swallows only ValueError. For each flag in raises, run a with-block that
    raises ValueError if the flag is true. Return the log.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(log_context, [
    (([False, True],), ['enter', 'exit', 'enter', 'exit']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class with `__enter__` and `__exit__`. Returning True suppresses the error.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`contextlib.contextmanager` with try/except/finally.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A class with `__enter__` and `__exit__`. Returning True suppresses the error.

```python
def log_context(raises):
    log = []

    class Managed:
        def __enter__(self):
            log.append("enter")
            return self

        def __exit__(self, exc_type, exc, tb):
            log.append("exit")
            return exc_type is ValueError
    for flag in raises:
        with Managed():
            if flag:
                raise ValueError("boom")
    return log
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `contextlib.contextmanager` with try/except/finally.

```python
def log_context(raises):
    import contextlib
    log = []

    @contextlib.contextmanager
    def managed():
        log.append("enter")
        try:
            yield
        except ValueError:
            pass
        finally:
            log.append("exit")
    for flag in raises:
        with managed():
            if flag:
                raise ValueError("boom")
    return log
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class with `__enter__` and `__exit__`. Returning True suppresses the error. That costs O(n) time and O(n) space."

"I can do better: `contextlib.contextmanager` with try/except/finally. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `raises = [False, True]`

1. managed() returns None
2. managed() returns None
3. managed() returns None
4. ValueError raised: boom
5. ValueError raised: boom
6. managed() returns None
7. log\_context() returns ['enter', 'exit', 'enter', 'exit']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/a-context-manager)

<a id="q17"></a>
### Q17. Method Resolution Order

🟡 Interview standard · *Diamond inheritance*

With class B(A), C(A), D(B, C), each defining who(), return D's MRO names, and the result of D().who() where each calls super and adds its name.

**Examples**

- `` → `(['D', 'B', 'C', 'A', 'object'], 'DBCA')`

In [ ]:
def mro_demo():
    """
    With class B(A), C(A), D(B, C), each defining who(), return D's MRO names,
    and the result of D().who() where each calls super and adds its name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mro_demo, [
    ((), (['D', 'B', 'C', 'A', 'object'], 'DBCA')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the classes and read `__mro__`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`D.mro()`. Super follows the MRO, not just the parent.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build the classes and read `__mro__`.

```python
def mro_demo():
    class A:
        def who(self):
            return "A"

    class B(A):
        def who(self):
            return "B" + super().who()

    class C(A):
        def who(self):
            return "C" + super().who()

    class D(B, C):
        def who(self):
            return "D" + super().who()
    names = []
    for k in D.__mro__:
        names.append(k.__name__)
    return (names, D().who())
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `D.mro()`. Super follows the MRO, not just the parent.

```python
def mro_demo():
    class A:
        def who(self):
            return "A"

    class B(A):
        def who(self):
            return "B" + super().who()

    class C(A):
        def who(self):
            return "C" + super().who()

    class D(B, C):
        def who(self):
            return "D" + super().who()
    return ([k.__name__ for k in D.mro()], D().who())
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the classes and read `__mro__`. That costs O(1) time and O(1) space."

"I can do better: `D.mro()`. Super follows the MRO, not just the parent. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: ``

1. A() returns None
2. B() returns &lt;cell at 0x10933e8f0: empty&gt;
3. C() returns &lt;cell at 0x10933e830: empty&gt;
4. D() returns &lt;cell at 0x10933f8e0: empty&gt;
5. who() returns 'A'
6. who() returns 'CA'
7. who() returns 'BCA'
8. who() returns 'DBCA'
9. mro\_demo() returns (['D', 'B', 'C', 'A', 'object'], 'DBCA')

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/method-resolution-order)

<a id="q18"></a>
### Q18. \_\_eq\_\_ and \_\_hash\_\_

🟡 Interview standard · *Objects in sets*

A Point with x and y: make equal points count as one in a set. Return the number of distinct points in a list of pairs.

**Examples**

- `pairs = [(1, 2), (1, 2), (3, 4)]` → `2`

In [ ]:
from dataclasses import dataclass, field

def distinct_points(pairs):
    """
    A Point with x and y: make equal points count as one in a set. Return the
    number of distinct points in a list of pairs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(distinct_points, [
    (([(1, 2), (1, 2), (3, 4)],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class defining `__eq__` and `__hash__` together.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`@dataclass(frozen=True)` generates both.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A class defining `__eq__` and `__hash__` together.

```python
def distinct_points(pairs):
    class Point:
        def __init__(self, x, y):
            self.x, self.y = x, y

        def __eq__(self, other):
            return (self.x, self.y) == (other.x, other.y)

        def __hash__(self):
            return hash((self.x, self.y))
    return len({Point(*p) for p in pairs})
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `@dataclass(frozen=True)` generates both.

```python
def distinct_points(pairs):
    @dataclass(frozen=True)
    class Point:
        x: int
        y: int
    return len({Point(*p) for p in pairs})
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class defining `__eq__` and `__hash__` together. That costs O(n) time and O(n) space."

"I can do better: `@dataclass(frozen=True)` generates both. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pairs = [(1, 2), (1, 2), (3, 4)]`

1. Point() returns None
2. \_\_annotate\_\_() returns {'x': &lt;class 'int'&gt;, 'y': &lt;class 'int'&gt;}
3. distinct\_points() returns 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/eq-and-hash)

<a id="q19"></a>
### Q19. LRU Cache With OrderedDict

🟡 Interview standard · *Stdlib power*

Replay operations on an LRU cache of the given capacity: ('put', k, v) and ('get', k). Return the results of each get (−1 if missing).

**Examples**

- `capacity = 2, ops = [('put', 1, 1), ('put', 2, 2), ('get', 1), ('put', 3, 3), ('get', 2), ('get', 3)]` → `[1, -1, 3]`

In [ ]:
def lru_replay(capacity, ops):
    """
    Replay operations on an LRU cache of the given capacity: ('put', k, v) and
    ('get', k). Return the results of each get (−1 if missing).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lru_replay, [
    ((2, [('put', 1, 1), ('put', 2, 2), ('get', 1), ('put', 3, 3), ('get', 2), ('get', 3)]), [1, -1, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A plain list of keys in recency order. Removal is O(n).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`OrderedDict.move_to_end` and `popitem(last=False)`: O(1).

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A plain list of keys in recency order. Removal is O(n).

```python
def lru_replay(capacity, ops):
    data, order, out = {}, [], []
    for op in ops:
        if op[0] == "get":
            if op[1] in data:
                order.remove(op[1])
                order.append(op[1])
                out.append(data[op[1]])
            else:
                out.append(-1)
        else:
            if op[1] in data:
                order.remove(op[1])
            elif len(data) == capacity:
                del data[order.pop(0)]
            data[op[1]] = op[2]
            order.append(op[1])
    return out
```

**Cost:** O(n) per op time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `OrderedDict.move_to_end` and `popitem(last=False)`: O(1).

```python
def lru_replay(capacity, ops):
    cache, out = OrderedDict(), []
    for op in ops:
        if op[0] == "get":
            if op[1] in cache:
                cache.move_to_end(op[1])
            out.append(cache.get(op[1], -1))
        else:
            cache[op[1]] = op[2]
            cache.move_to_end(op[1])
            if len(cache) > capacity:
                cache.popitem(last=False)
    return out
```

**Cost:** O(1) per op time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A plain list of keys in recency order. Removal is O(n). That costs O(n) per op time and O(c) space."

"I can do better: `OrderedDict.move_to_end` and `popitem(last=False)`: O(1). That's O(1) per op time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/lru-cache-with-ordereddict)

<a id="q20"></a>
### Q20. Coroutines by Hand

🔴 Stretch · *What async does underneath*

Each task is a generator that yields when it would wait. Run tasks round robin until all finish, and return the order of events. This is how an event loop works.

**Examples**

- `specs = [('a', 2), ('b', 1)]` → `['a0', 'b0', 'a1']`

In [ ]:
def round_robin(specs):
    """
    Each task is a generator that yields when it would wait. Run tasks round
    robin until all finish, and return the order of events. This is how an event
    loop works.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(round_robin, [
    (([('a', 2), ('b', 1)],), ['a0', 'b0', 'a1']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of tasks, removing finished ones.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `deque`: pop left, run one step, push back.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A list of tasks, removing finished ones.

```python
def round_robin(specs):
    log = []

    def task(name, steps):
        for i in range(steps):
            log.append(f"{name}{i}")
            yield
    tasks = [task(n, s) for n, s in specs]
    while tasks:
        for t in list(tasks):
            try:
                next(t)
            except StopIteration:
                tasks.remove(t)
    return log
```

**Cost:** O(total) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A `deque`: pop left, run one step, push back.

```python
def round_robin(specs):
    log = []

    def task(name, steps):
        for i in range(steps):
            log.append(f"{name}{i}")
            yield
    ready = deque(task(n, s) for n, s in specs)
    while ready:
        t = ready.popleft()
        try:
            next(t)
            ready.append(t)
        except StopIteration:
            pass
    return log
```

**Cost:** O(total) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of tasks, removing finished ones. That costs O(total) time and O(n) space."

"I can do better: A `deque`: pop left, run one step, push back. That's O(total) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `specs = [('a', 2), ('b', 1)]`

1. task() returns None
2. task() returns None
3. task() returns None
4. task() returns None
5. StopIteration raised: 
6. task() returns None
7. StopIteration raised: 
8. round\_robin() returns ['a0', 'b0', 'a1']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/python-advanced/coroutines-by-hand)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the GIL?</b></summary>

A lock in standard CPython that lets one thread run Python bytecode at a time. CPU-heavy pure-Python code should use processes; I/O-heavy code does well with threads or asyncio.

</details>

<details>
<summary><b>2. Threads vs processes vs asyncio?</b></summary>

Threads suit I/O with blocking libraries. Processes suit CPU-heavy work. Asyncio suits many concurrent waiting tasks in one thread, using libraries that support it.

</details>

<details>
<summary><b>3. What is a decorator?</b></summary>

A function that takes a function and returns a new one, adding behaviour. `@d` above a definition means `f = d(f)`.

</details>

<details>
<summary><b>4. Generator vs list?</b></summary>

A generator produces values lazily and holds only its state, so it can handle huge or endless streams. It can be consumed only once, and it can't be indexed.

</details>

<details>
<summary><b>5. Iterable vs iterator?</b></summary>

An iterable can give you an iterator through `__iter__`. An iterator produces values through `__next__`, and it remembers its position.

</details>

<details>
<summary><b>6. Shallow vs deep copy?</b></summary>

A shallow copy makes a new outer container that shares inner objects. A deep copy recursively copies everything.

</details>

<details>
<summary><b>7. == vs is?</b></summary>

`==` compares values, while `is` checks object identity. Use `is` for None.

</details>

<details>
<summary><b>8. What are \*args and \*\*kwargs?</b></summary>

They collect extra positional arguments into a tuple and extra keyword arguments into a dictionary.

</details>

<details>
<summary><b>9. @classmethod vs @staticmethod?</b></summary>

A classmethod receives the class, so it suits alternative constructors. A staticmethod receives nothing automatically, so it's a plain function that lives in the class.

</details>

<details>
<summary><b>10. What is the MRO?</b></summary>

The order Python searches classes for a method, computed by C3 linearisation. `super()` moves to the next class in it.

</details>

<details>
<summary><b>11. How do dictionaries work?</b></summary>

A hash table: the key's hash picks a slot, and equality confirms the match. Lookups are O(1) on average, and insertion order has been preserved since Python 3.7.

</details>

<details>
<summary><b>12. How does Python's sort work?</b></summary>

Timsort, a hybrid of merge and insertion sort: O(n log n), stable, and fast on partly ordered data.

</details>

<details>
<summary><b>13. What is duck typing?</b></summary>

Code cares what an object can do, not what class it is. If it has the method you call, it works. Type hints document intent, and they aren't enforced at run time.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Names, objects and the traps they cause
- [ ] I can explain: Decorators, from nothing
- [ ] I can explain: Iterators and generators
- [ ] I can explain: Context managers and classes
- [ ] I can explain: Know your standard library
- [ ] I can explain: Concurrency: GIL, threads, processes and async

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) A Context Manager
- [ ] [Q17](#q17) Method Resolution Order
- [ ] [Q18](#q18) \_\_eq\_\_ and \_\_hash\_\_
- [ ] [Q19](#q19) LRU Cache With OrderedDict
- [ ] [Q20](#q20) Coroutines by Hand

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.